# 🌾 Crop Recommendation RAG System — End-to-End Notebook

**Assignment: Associate AI Engineer | RAG System**

This notebook walks through the complete pipeline:
1. Dataset exploration & statistics
2. Chunking strategy comparison (row vs aggregate vs hybrid)
3. Vector indexing with two embedding models
4. Retrieval experiments (Semantic / BM25 / Hybrid + RRF)
5. Query rewriting for agronomic shorthand
6. Grounded answer generation with Claude
7. Evaluation: Retrieval metrics + LLM-as-judge


In [ ]:
import os, sys, warnings
sys.path.insert(0, '.')   # run from repo root
warnings.filterwarnings('ignore')

# Set your Anthropic API key
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-YOUR_KEY_HERE'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('Setup complete')


## 1. Dataset Exploration

In [ ]:
df = pd.read_csv('data/Crop_recommendation.csv')
print(f'Shape: {df.shape}')
print(f'Crops ({df["label"].nunique()}): {sorted(df["label"].unique())}')
df.describe().round(2)


In [ ]:
numeric_cols = ['N','P','K','temperature','humidity','ph','rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    df.groupby('label')[col].mean().sort_values().plot(
        kind='barh', ax=axes[i], title=f'Mean {col}', color='steelblue')
axes[-1].set_visible(False)
plt.suptitle('Mean Feature Values per Crop', fontsize=14)
plt.tight_layout()
plt.savefig('evaluation/feature_distributions.png', dpi=100, bbox_inches='tight')
plt.show()


## 2. Chunking Strategies

| Strategy | Docs | Pros | Cons |
|----------|------|------|------|
| **row** | 2200 | Granular exact values | Noisy, redundant |
| **aggregate** | 22 | Compact, summary stats | Loses outlier info |
| **hybrid** | 242 | Balance precision + coverage | Slightly larger index |

**Choice: Hybrid** — aggregate docs handle general crop profile questions while sampled row docs help with specific numeric-threshold queries.


In [ ]:
from src.ingestion import load_and_chunk

for strategy in ['row', 'aggregate', 'hybrid']:
    docs = load_and_chunk('data/Crop_recommendation.csv', strategy=strategy)
    print(f'{strategy:12s}: {len(docs):5d} docs  |  {docs[0]["text"][:90]}...')

docs = load_and_chunk('data/Crop_recommendation.csv', strategy='hybrid')
print('\n=== AGGREGATE DOC ===')
print(docs[0]['text'])
print('\n=== ROW-LEVEL DOC ===')
print(docs[-1]['text'])


## 3. Embedding & Indexing

| Model | Dims | Speed | Quality |
|-------|------|-------|---------|
| `all-MiniLM-L6-v2` | 384 | Fast | Good |
| `all-mpnet-base-v2` | 768 | Slower | Better |

**Production choice: MiniLM** — 3x faster indexing, lower memory, adequate recall for this dataset size.


In [ ]:
import time
from src.vector_store import VectorStore

docs = load_and_chunk('data/Crop_recommendation.csv', strategy='hybrid')

t0 = time.time()
vs = VectorStore(collection_name='notebook_demo', model_key='minilm',
                 persist_dir='./chroma_notebook')
vs.index(docs)
print(f'Indexed {len(docs)} docs in {time.time()-t0:.1f}s')


## 4. Retrieval Experiments

In [ ]:
from src.retriever import Retriever, rewrite_query

# Query rewriting demo
for q in ['What grows in dry weather?', 'Crops for humid tropical climate', 'acidic soil low N']:
    print(f'Before: {q}')
    print(f'After : {rewrite_query(q)}')
    print()


In [ ]:
query = 'Which crop needs high humidity and very little rainfall?'

ret_sem  = Retriever(vs, docs, mode='semantic', use_rewriter=True)
ret_bm25 = Retriever(vs, docs, mode='bm25',     use_rewriter=False)
ret_hyb  = Retriever(vs, docs, mode='hybrid',   use_rewriter=True)

for label, ret in [('Semantic', ret_sem), ('BM25', ret_bm25), ('Hybrid', ret_hyb)]:
    hits = ret.retrieve(query, k=5)
    crops = [h['metadata']['crop'] for h in hits]
    scores = [round(h['score'], 3) for h in hits]
    print(f'{label:10s}: crops={crops}')
    print(f'           scores={scores}')


## 5. Generation with Claude

In [ ]:
from src.generator import generate_answer

ret_hyb = Retriever(vs, docs, mode='hybrid', use_rewriter=True)

sample_questions = [
    'What crop grows best with high nitrogen, low rainfall, and acidic soil?',
    'Which crops are suitable for a humid tropical climate?',
    'Compare nitrogen requirements for rice vs wheat.',
    'What should I grow if my soil pH is 5.5 and rainfall is 250mm?',
]

for q in sample_questions:
    hits = ret_hyb.retrieve(q, k=5)
    result = generate_answer(q, hits, api_key=os.environ['ANTHROPIC_API_KEY'])
    print(f'Q: {q}')
    print(f'A: {result["answer"][:350]}...')
    print(f'   tokens in={result["input_tokens"]} out={result["output_tokens"]}')
    print()


In [ ]:
# Metadata-filtered search (only rice rows)
q = 'What pH does rice need?'
hits_rice = vs.search(q, k=5, metadata_filter={'crop': 'rice'})
result = generate_answer(q, hits_rice, api_key=os.environ['ANTHROPIC_API_KEY'])
print('=== Metadata Filter: crop=rice ===')
print(result['answer'])


## 6. Evaluation

In [ ]:
from evaluation.evaluator import (
    TEST_QA_PAIRS, compute_retrieval_metrics,
    extract_crops_from_hits, llm_judge
)

print(f'Test set size: {len(TEST_QA_PAIRS)} Q&A pairs\n')
for qa in TEST_QA_PAIRS[:3]:
    print(f'[{qa["id"]}] {qa["question"]}')
    print(f'      Expected: {qa["expected_crops"]}')


In [ ]:
# Retrieval metrics over first 15 questions
ret_hyb = Retriever(vs, docs, mode='hybrid', use_rewriter=True)
rows = []
for qa in TEST_QA_PAIRS[:15]:
    hits = ret_hyb.retrieve(qa['question'], k=5)
    crops = extract_crops_from_hits(hits)
    m = compute_retrieval_metrics(crops, qa['expected_crops'], k=5)
    rows.append({'id': qa['id'], **m})

res_df = pd.DataFrame(rows)
print(f'Mean Hit Rate  @ 5: {res_df.hit_rate.mean():.3f}')
print(f'Mean Precision@ 5: {res_df.precision_k.mean():.3f}')
print(f'Mean Recall   @ 5: {res_df.recall_k.mean():.3f}')
print(f'Mean MRR         : {res_df.rr.mean():.3f}')
res_df


In [ ]:
# LLM-as-judge on 3 samples
judge_rows = []
for qa in TEST_QA_PAIRS[:3]:
    hits = ret_hyb.retrieve(qa['question'], k=5)
    gen  = generate_answer(qa['question'], hits, api_key=os.environ['ANTHROPIC_API_KEY'])
    ctx  = ' '.join(h['text'] for h in hits)
    sc   = llm_judge(qa['question'], qa['ground_truth'], gen['answer'], ctx,
                     api_key=os.environ['ANTHROPIC_API_KEY'])
    judge_rows.append({'id': qa['id'], **sc})
    print(f'{qa["id"]}: faithfulness={sc.get("faithfulness",0):.2f}  '
          f'relevance={sc.get("relevance",0):.2f}  '
          f'correctness={sc.get("correctness",0):.2f}  '
          f'hallucination={sc.get("hallucination",0):.2f}')

pd.DataFrame(judge_rows)


## 7. Full Evaluation & Experiments

```bash
# Run all chunking/embedding/retrieval experiments (retrieval-only, fast)
python -m evaluation.experiments

# Full 27-question evaluation with LLM judge
python cli.py --evaluate --api-key $ANTHROPIC_API_KEY

# Interactive Q&A
python cli.py --interactive --api-key $ANTHROPIC_API_KEY

# Single query
python cli.py --query "What crop needs low rainfall and high potassium?" --api-key $ANTHROPIC_API_KEY
```


## System Architecture

```
CSV (2200 rows, 22 crops)
         │
    ┌────▼────────────────┐
    │   Ingestion           │
    │   hybrid chunking     │
    │   242 documents       │
    └────┬────────────────┘
         │
    ┌────▼────────────────┐
    │   Embeddings          │
    │   all-MiniLM-L6-v2   │
    └────┬────────────────┘
         │
    ┌────▼────────────────┐
    │   ChromaDB            │
    │   cosine similarity   │
    └────┬────────────────┘
         │
    ┌────▼────────────────┐
    │   Retriever           │
    │   BM25 + Semantic     │
    │   RRF fusion, k=5     │
    │   Query rewriting     │
    └────┬────────────────┘
         │ top-5 chunks
    ┌────▼────────────────┐
    │   Generator           │
    │   Claude Sonnet 4.6   │
    │   Grounded + Cited    │
    └─────────────────────┘
```
